In [2]:
import sionna.rt as rt
import numpy as np
import matplotlib.pyplot as plt

relay_scene = rt.load_scene(
    rt.scene.munich,
    merge_shapes=False
)

relay_scene.frequency = 3.5e9

relay_scene.tx_array = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization = "V"
)

relay_scene.rx_array = relay_scene.tx_array

In [3]:
initial_positions = {
    "base_tx": [495.777, 211.072, 19.704],
    "relay": [225.087, -202.369, 50.0],
    "user_a": [46.182, 84.560, 1.5],
    "user_b": [28.599, 225.094, 1.5],
    "user_c": [-537.994, -398.940, 1.5]
}

base_tx = rt.Transmitter(
    name = "base_tx",
    position=initial_positions["base_tx"],
    power_dbm=30,
    display_radius=4
)

relay_rx = rt.Receiver(
    name = "relay_rx",
    position=initial_positions["relay"],
    display_radius=5
)

relay_tx = rt.Transmitter(
    name = "relay_tx",
    position=initial_positions["relay"],
    power_dbm=20,
    display_radius=3
)

users = {}

for name in ["user_a", "user_b", "user_c"]:
    users[name] = rt.Receiver(
        name = name,
        position=initial_positions[name],
        display_radius=4
    )

devices = [
    base_tx,
    relay_tx,
    relay_rx,
    *users.values()
]

for device in devices:
    if relay_scene.get(device.name) is not None:
        relay_scene.remove(device.name)

    relay_scene.add(devices)

print("Transmitters:", list(relay_scene.transmitters))
print("Receivers:", list(relay_scene.receivers))

relay_scene.preview()

Transmitters: ['base_tx', 'relay_tx']
Receivers: ['relay_rx', 'user_a', 'user_b', 'user_c']


In [4]:
if relay_scene.get("relay_tx") is not None:
    relay_scene.remove("relay_tx")

print("Передавачі:", list(relay_scene.transmitters))
print("Приймачі:", list(relay_scene.receivers))

Передавачі: ['base_tx']
Приймачі: ['relay_rx', 'user_a', 'user_b', 'user_c']


In [5]:
relay_solver = rt.PathSolver()

base_los_paths = relay_solver(
    scene=relay_scene,
    max_depth=0,
    los = True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False
)

In [6]:
base_receiver_names = list(relay_scene.receivers.keys())
valid = np.array(base_los_paths.valid)

for i, name in enumerate(base_receiver_names):
    has_los = bool(valid[i].any())
    print(f"base_tx -> {name}: LOS = {has_los}")

base_tx -> relay_rx: LOS = True
base_tx -> user_a: LOS = False
base_tx -> user_b: LOS = False
base_tx -> user_c: LOS = False


In [7]:
relay_scene.preview(paths=base_los_paths)

In [8]:
base_reflection_paths = relay_solver(
    scene=relay_scene,
    max_depth=2,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False
)

In [9]:
valid = np.array(base_reflection_paths.valid)

for i, name in enumerate(base_receiver_names):
    path_count = np.count_nonzero(valid[i])
    print(f"base_tx -> {name}, find paths - {path_count}")

base_tx -> relay_rx, find paths - 2
base_tx -> user_a, find paths - 0
base_tx -> user_b, find paths - 0
base_tx -> user_c, find paths - 0


In [10]:
relay_scene.preview(paths=base_reflection_paths)

In [11]:
for name in ["base_tx", "relay_rx"]:
    if relay_scene.get(name) is not None:
        relay_scene.remove(name)

if relay_scene.get("relay_tx") is None:
    relay_scene.add(relay_tx)

print("Transmitters:", list(relay_scene.transmitters))
print("Receivers:", list(relay_scene.receivers))

Transmitters: ['relay_tx']
Receivers: ['user_a', 'user_b', 'user_c']


In [12]:
relay_los_paths = relay_solver(
    scene=relay_scene,
    max_depth=0,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False
)

In [13]:
relay_receiver_names = list(relay_scene.receivers.keys())
valid = np.array(relay_los_paths.valid)

for i, name in enumerate(relay_receiver_names):
    has_los = bool(valid[i].any())
    print(f"relaey_tx -> {name}: LOS = {has_los}")

relaey_tx -> user_a: LOS = False
relaey_tx -> user_b: LOS = False
relaey_tx -> user_c: LOS = False


In [14]:
relay_scene.preview(paths=relay_los_paths)

In [15]:
relay_refraction_paths = relay_solver(
    scene=relay_scene,
    max_depth=2,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False
)

In [16]:
valid = np.array(relay_refraction_paths.valid)

for i, name in enumerate(relay_receiver_names):
    count = np.count_nonzero(valid[i])
    print(f"relay_tx -> {name}: find paths = {count}")

relay_tx -> user_a: find paths = 1
relay_tx -> user_b: find paths = 0
relay_tx -> user_c: find paths = 0


In [17]:
relay_scene.preview(paths=relay_refraction_paths)

In [18]:
a_base, _ = base_reflection_paths.cir(
    normalize_delays=False,
    out_type = "numpy"
)

a_relay, _ = relay_refraction_paths.cir(
    normalize_delays=False,
    out_type = "numpy"
)

relay_index = base_receiver_names.index("relay_rx")
user_index = relay_receiver_names.index("user_a")

base_to_relay = a_base[relay_index, 0, 0, 0, :, 0]
relay_to_user = a_relay[user_index, 0, 0, 0, :, 0]

print("База → ретранслятор:", base_to_relay)
print("Форма:", base_to_relay.shape)

print("Ретранслятор → user_a:", relay_to_user)
print("Форма:", relay_to_user.shape)

База → ретранслятор: [ 6.6483963e-06-1.20556215e-05j -2.4296999e-06+1.34998600e-05j]
Форма: (2,)
Ретранслятор → user_a: [4.0427634e-07+1.255022e-06j]
Форма: (1,)


In [19]:
gain_base_to_relay = np.sum(np.abs(base_to_relay) ** 2)
gain_relay_to_user = np.sum(np.abs(relay_to_user) ** 2)

base_power_w = 1.0
relay_power_w = 0.1

received_at_relay_w = base_power_w * gain_base_to_relay
received_at_user_w = relay_power_w * gain_relay_to_user

print(
    "Base -> repeater:",
    10 * np.log10(received_at_relay_w / 1e-3),
    "dBm"
)

print(
    "Repeater -> user_a:",
    10 * np.log10(received_at_user_w / 1e-3),
    "dBm"
)

Base -> repeater: -64.22866 dBm
Repeater -> user_a: -97.598206 dBm


In [20]:
bandwidth_hz = 20e6
noise_figure_db = 7.0

noise_dbm = (
    -174
    + 10 * np.log10(bandwidth_hz)
    + noise_figure_db
)

received_at_relay_dbm = 10 * np.log10(received_at_relay_w / 1e-3)
received_at_user_dbm = 10 * np.log10(received_at_user_w / 1e-3)

snr_base_to_relay_db = received_at_relay_dbm - noise_dbm
snr_relay_to_user_db = received_at_user_dbm - noise_dbm

print(f"noise of receiver: {noise_dbm:.2f} dbm")
print(f"SNR base -> repeater: {snr_base_to_relay_db:.2f} dB")
print(f"SNR repeater -> user_a: {snr_relay_to_user_db:.2f} dB")

noise of receiver: -93.99 dbm
SNR base -> repeater: 29.76 dB
SNR repeater -> user_a: -3.61 dB


In [21]:
relay_power_test_dbm = 30
relay_power_test_w = 1e-3 * 10 ** (relay_power_test_dbm / 10)

received_test_w = relay_power_test_w * gain_relay_to_user
received_test_dbm = 10 * np.log10(received_test_w / 1e-3)

snr_test_db = received_test_dbm - noise_dbm

print("User_a signal:", received_test_dbm, "dbm")
print("User_a SNR:", snr_test_db, "db")

User_a signal: -87.598206 dbm
User_a SNR: 6.3914944769539375 db


In [22]:
new_relay_position = (
    np.array(initial_positions["relay"]) + [0, 0, 30]
).tolist()

relay_tx.position = new_relay_position
relay_rx.position = new_relay_position

In [23]:
relay_los_paths = relay_solver(
    scene = relay_scene,
    max_depth = 0,
    los = True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False
)

valid = np.array(relay_los_paths.valid)

for i, name in enumerate(relay_receiver_names):
    print(f"relay_tx -> {name}: LOS = {bool(valid[i].any())}")

relay_tx -> user_a: LOS = True
relay_tx -> user_b: LOS = False
relay_tx -> user_c: LOS = False


In [24]:
a_relay_80, _ = relay_los_paths.cir(
    normalize_delays=False,
    out_type="numpy"
)

user_index = relay_receiver_names.index("user_a")

coefficients_80 = a_relay_80[user_index, 0, 0, 0, :, 0]
gain_80 = np.sum(np.abs(coefficients_80) ** 2)

relay_power_w = 0.1
relay_tx.power_dbm = 20

received_80_w = relay_power_w * gain_80
received_80_dbm = 10 * np.log10(received_80_w / 1e-3)

snr_80_db = received_80_dbm - noise_dbm

print(f"user_a signal in 80m: {received_80_dbm:.2f} dBm")
print(f"SNR in 80m: {snr_80_db:.2f} db")

user_a signal in 80m: -74.14 dBm
SNR in 80m: 19.85 db


In [25]:
received_50_dbm = -97.598206

improvement_db = received_80_dbm - received_50_dbm
print(f"Difference between 80m and 50m: {improvement_db:+.2f} db")

Difference between 80m and 50m: +23.46 db


In [26]:
if relay_scene.get("relay_tx") is not None:
    relay_scene.remove("relay_tx")

relay_rx.position = np.array(relay_tx.position).reshape(3).tolist()

if relay_scene.get("base_tx") is None:
    relay_scene.add(base_tx)

if relay_scene.get("relay_rx") is None:
    relay_scene.add(relay_rx)

base_receiver_names_80 = list(relay_scene.receivers.keys())

print("Transmitters:", list(relay_scene.transmitters))
print("Receivers", base_receiver_names_80)
print("Repeater position:", relay_rx.position)

Transmitters: ['base_tx']
Receivers ['user_a', 'user_b', 'user_c', 'relay_rx']
Repeater position: [[225.087, -202.369, 80]]


In [27]:
base_paths_80 = relay_solver(
    scene = relay_scene,
    max_depth=2,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False
)

a_base_80, _ = base_paths_80.cir(
    normalize_delays=False,
    out_type="numpy"
)

relay_index_80 = base_receiver_names_80.index("relay_rx")
coefficients_base_80 = a_base_80[relay_index_80, 0, 0, 0, :, 0]

gain_base_80 = np.sum(np.abs(coefficients_base_80) ** 2)
received_at_relay_80_w = 1.0 * gain_base_80

if received_at_relay_80_w > 0:
    received_at_relay_80_dbm = 10 * np.log10(
        received_at_relay_80_w / 1e-3
    )

    snr_base_80_db = received_at_relay_80_dbm - noise_dbm

    print(f"Base -> repeater: {received_at_relay_80_dbm:.2f} dbm")
    print(f"SNR: {snr_base_80_db:.2f} db")
else:
    print("Received w equals to 0, u need to check paths")

Base -> repeater: -64.27 dbm
SNR: 29.72 db
